# Demo — Student Performance Early-Warning Model (EDU-02)

This notebook takes **one student's early-course data** (first ~25% of a course:
demographics, online activity, early assignment results) and returns:

- a **risk probability** (0–1) that the student will fail or withdraw,
- a **risk band** (Low / Medium / High),
- whether the student is **flagged for an advisor check-in**,
- simple **signals** explaining what looks concerning.

**How to run (Google Colab or locally):**

1. Clone the repository and open this notebook from its root
   (in Colab: `File → Open notebook → GitHub`, or run the clone cell below).
2. Run all cells top to bottom (`Runtime → Run all`). No dataset download is
   needed: the trained model ships with the repository (`models/`, under 1 MB),
   and the setup cell installs only the four inference dependencies from
   `requirements-demo.txt` (a few seconds).

The model was trained on the 2013 cohorts of OULAD and evaluated once on the
unseen 2014J cohort: recall 0.76 / precision 0.58 / PR-AUC 0.71 for the
at-risk class (details: `reports/final_evaluation.md`).

In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Mubina-lazy/edu02-early-warning-model.git"
REPO_DIR = "edu02-early-warning-model"


def at_repo_root() -> bool:
    return Path("src/predict.py").exists() and Path("models/final_model.joblib").exists()


if IN_COLAB:
    if not at_repo_root() and not Path(REPO_DIR).exists():
        !git clone -q {REPO_URL}
    if not at_repo_root():
        %cd {REPO_DIR}
    # A Colab session can still hold a clone from an earlier run, and scoring
    # with a stale artifact is exactly the hidden-state problem a reproducible
    # demo must avoid. Reset to the published main every time (fetch+reset
    # rather than pull, because the published history has been rewritten).
    !git fetch -q origin main && git reset -q --hard origin/main

# Only the inference dependencies: pandas, scikit-learn, xgboost, joblib.
# requirements.txt (training + MLflow) is not needed to score a student.
%pip install -q -r requirements-demo.txt

sys.path.insert(0, str(Path("src").resolve()))
from predict import load_model, predict_risk

# load_model() compares the installed library versions against the ones that
# fitted the pipeline and warns if they differ.
model, meta = load_model()
print(f"Loaded final model: {meta['run_name']} "
      f"(decision threshold {meta['threshold']:.3f}, tuned on validation)")
print(f"Fitted with: {meta.get('fitted_with', 'not recorded in this artifact')}")

Note: you may need to restart the kernel to use updated packages.


Loaded final model: E4_xgboost (decision threshold 0.327, tuned on validation)
Fitted with: {'scikit-learn': '1.9.0', 'xgboost': '3.2.0', 'pandas': '2.3.3'}


## Input format

One student = one Python dict. Activity numbers cover **only the early window**
(up to ~day 60–67 of the course). `early_tma_mean_score = None` means the
student has not submitted any assignment yet (that is information, not an error).

| field | example | meaning |
|---|---|---|
| `code_module` | `"AAA"` | course code (AAA–GGG) |
| `gender`, `region`, `highest_education`, `imd_band`, `age_band`, `disability` | `"F"`, `"Scotland"`, ... | demographics as in OULAD |
| `early_total_clicks` | `3` | clicks in the learning environment so far |
| `early_active_days` | `1` | days with any activity |
| `days_since_last_activity` | `57` | days since last click (None = never active) |
| `early_tma_due_count` / `early_tma_submitted_count` | `2` / `0` | assignments due vs submitted |
| `early_tma_mean_score` | `None` | average score of submitted assignments |
| `date_registration` | `-144` | registration day relative to course start |
| `num_of_prev_attempts`, `studied_credits` | `1`, `60` | history and load |

## Example 1 — a disengaged student (real unseen row, 2014J cohort)

Registered early but clicked only 3 times on a single day, then disappeared for
57 days, and never submitted either of the two assignments that were already due.
The true outcome of this student was **Withdrawn**.

In [2]:
student_disengaged = {
    "code_module": "AAA", "gender": "F", "region": "East Anglian Region",
    "highest_education": "A Level or Equivalent", "imd_band": "70-80%",
    "age_band": "0-35", "disability": "False",
    "early_total_clicks": 3, "early_active_days": 1,
    "days_since_last_activity": 57,
    "early_tma_due_count": 2, "early_tma_submitted_count": 0,
    "early_tma_mean_score": None,
    "date_registration": -144, "num_of_prev_attempts": 1, "studied_credits": 60,
}

result = predict_risk(student_disengaged, model, meta)
for k, v in result.items():
    print(f"{k}: {v}")

risk_probability: 0.889
risk_band: High
flagged_for_advisor: True
decision_threshold: 0.327
top_factors: [{'factor': 'days since last activity', 'direction': 'increases risk', 'contribution': 1.57}, {'factor': 'assignments submitted', 'direction': 'increases risk', 'contribution': 0.927}, {'factor': 'code_module = AAA', 'direction': 'lowers risk', 'contribution': -0.568}]
signals: ['very low online activity (bottom quartile)', 'inactive for 57 days at the check point', 'has not submitted any assignment that was already due', 'has previous unsuccessful attempts at this course']
note: Decision-support only: an advisor must review every flag.


## Example 2 — an engaged student (real unseen row, 2014J cohort)

1,101 clicks over 45 active days, both due assignments submitted, average
score 85. The true outcome of this student was **Pass**.

In [3]:
student_engaged = {
    "code_module": "AAA", "gender": "F", "region": "East Anglian Region",
    "highest_education": "A Level or Equivalent", "imd_band": "60-70%",
    "age_band": "35-55", "disability": "False",
    "early_total_clicks": 1101, "early_active_days": 45,
    "days_since_last_activity": 8,
    "early_tma_due_count": 2, "early_tma_submitted_count": 2,
    "early_tma_mean_score": 85.0,
    "date_registration": -38, "num_of_prev_attempts": 0, "studied_credits": 60,
}

result = predict_risk(student_engaged, model, meta)
for k, v in result.items():
    print(f"{k}: {v}")

risk_probability: 0.122
risk_band: Low
flagged_for_advisor: False
decision_threshold: 0.327
top_factors: [{'factor': 'average early assignment score', 'direction': 'lowers risk', 'contribution': -0.64}, {'factor': 'assignments submitted', 'direction': 'lowers risk', 'contribution': -0.351}, {'factor': 'code_module = AAA', 'direction': 'lowers risk', 'contribution': -0.335}]
signals: ['no obvious warning signals - risk driven by weaker patterns']
note: Decision-support only: an advisor must review every flag.


## Input validation — bad inputs fail loudly and clearly

The prediction function checks the input **before** it reaches the model:
missing required fields, impossible values, or inconsistent counts produce a
readable error instead of a silent wrong prediction.

In [4]:
bad_inputs = {
    "missing required field": {**student_engaged},          # will drop a field below
    "score above 100": {**student_engaged, "early_tma_mean_score": 150},
    "negative clicks": {**student_engaged, "early_total_clicks": -5},
    "submitted more than due": {**student_engaged,
                                "early_tma_submitted_count": 5},
    "unknown course code": {**student_engaged, "code_module": "ZZZ"},
}
del bad_inputs["missing required field"]["early_total_clicks"]

for name, bad in bad_inputs.items():
    try:
        predict_risk(bad, model, meta)
        print(f"{name}: NOT CAUGHT <- this would be a bug")
    except ValueError as e:
        print(f"{name}:\n   {e}\n")

missing required field:
   invalid input: missing required field: 'early_total_clicks'

score above 100:
   invalid input: 'early_tma_mean_score' = 150.0 is above the maximum 100

negative clicks:
   invalid input: 'early_total_clicks' = -5.0 is below the minimum 0

submitted more than due:
   invalid input: submitted TMA count cannot exceed the number due

unknown course code:
   invalid input: 'code_module' must be one of ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']



## Edge case — a student with zero activity

A student who registered but never opened the course. The pipeline treats this
as a valid (and very alarming) state — not as missing data.

In [5]:
student_ghost = {
    "code_module": "BBB", "gender": "M", "region": "London Region",
    "highest_education": "Lower Than A Level", "imd_band": "Missing",
    "age_band": "0-35", "disability": "False",
    "early_total_clicks": 0, "early_active_days": 0,
    "days_since_last_activity": None,   # never active
    "early_tma_due_count": 2, "early_tma_submitted_count": 0,
    "early_tma_mean_score": None,
    "date_registration": -10, "num_of_prev_attempts": 0, "studied_credits": 120,
}

result = predict_risk(student_ghost, model, meta)
for k, v in result.items():
    print(f"{k}: {v}")

risk_probability: 0.964
risk_band: High
flagged_for_advisor: True
decision_threshold: 0.327
top_factors: [{'factor': 'days active in the early window', 'direction': 'increases risk', 'contribution': 0.96}, {'factor': 'assignments submitted', 'direction': 'increases risk', 'contribution': 0.831}, {'factor': 'no assignment submitted yet', 'direction': 'increases risk', 'contribution': 0.607}]
signals: ['no online activity at all in the early window', 'has not submitted any assignment that was already due']
note: Decision-support only: an advisor must review every flag.


## How to read the output

- **risk_probability** — the model's estimate that the student will fail or
  withdraw; **risk_band** — Low (< 0.327) / Medium (0.327–0.6) / High (≥ 0.6).
- **flagged_for_advisor** — True when the probability crosses the operating
  threshold (0.327, tuned on validation to catch ~84% of at-risk students).
- **top_factors** — what the *model itself* weighted most for this student,
  computed with TreeSHAP over the trained trees. A positive contribution
  pushed the risk up, a negative one pulled it down. This is the "top
  contributing factors" output the project brief asked for.
- **signals** — plain-language observations read straight off the input. They
  describe the student's situation, not the model's internal weights, so an
  advisor can act on them without reading the model.

**Important limitations** (full list in the README): trained on UK Open
University data from 2013–2014 — a methodology prototype, **not** a system
ready for another institution; the model misses at-risk students whose
problems start after the first quarter of the course; every flag requires
human review, and the output must never drive automatic decisions.